# TrueID Live Commerce Copilot - Full Demo

Run the setup cell once, then use the UI cells for the full demo, catalog manager, and live stream demos.


In [ ]:
import contextlib
import io
import os
import sys
import subprocess
from pathlib import Path

from IPython.display import HTML, display

REPO_URL = "https://github.com/Siratish/Live-Commerce-Copilot.git"
REPO_DIR_NAME = "Live-Commerce-Copilot"


def setup_card(title, detail=""):
    display(HTML(
        '<div style="border:1px solid #bfdbfe;background:#eff6ff;color:#1e3a8a;'
        'border-radius:8px;padding:12px;font-family:Arial,sans-serif;margin-bottom:10px;">'
        f'<strong>{title}</strong>'
        f'<div style="font-size:13px;line-height:1.45;margin-top:4px;">{detail}</div>'
        '</div>'
    ))


def looks_like_project(path: Path) -> bool:
    return (path / "config" / "demo.yaml").exists() and (path / "src").exists()


repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_project(candidate):
        repo_root = candidate
        break

if repo_root is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
    repo_root = clone_parent / REPO_DIR_NAME
    if repo_root.exists() and not looks_like_project(repo_root):
        raise RuntimeError(f"{repo_root} exists but does not look like the target project.")
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)], stdout=subprocess.DEVNULL)

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

requirements = repo_root / "requirements.txt"
if requirements.exists():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
    )

try:
    import ipywidgets  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])

from src.utils.live_mic import preload_openai_whisper_model

with contextlib.redirect_stdout(io.StringIO()):
    PRELOADED_OPENAI_WHISPER_TURBO = preload_openai_whisper_model("turbo")

setup_card(
    "Notebook setup complete",
    f"Repo ready at <code>{repo_root}</code>. OpenAI Whisper turbo is loaded for the demo.",
)


In [ ]:
from pathlib import Path
from src.utils.full_demo_ui import display_full_demo_ui

display_full_demo_ui(Path.cwd())


## Catalog manager

Run this cell when you want to review products or promotions separately from the full demo. Products and promotions are shown one list at a time, and product additions can include an uploaded image.


In [ ]:
# CATALOG_MANAGER_CELL_V1
from pathlib import Path
from src.utils.full_demo_ui import display_catalog_manager_ui

display_catalog_manager_ui(Path.cwd())


## Live mode: audio-file stream

Run this cell to simulate a live stream from Audio 1, Audio 2, or an uploaded audio file. The stream panel releases chunks to ASR only after playback reaches each chunk.


In [ ]:
from pathlib import Path
from src.utils.full_demo_ui import display_live_audio_file_demo_ui

display_live_audio_file_demo_ui(Path.cwd())


## Live mode: microphone stream

Run this cell to stream speech from the browser microphone into ASR and commerce actions.


In [ ]:
from pathlib import Path
from src.utils.full_demo_ui import display_live_mic_demo_ui

display_live_mic_demo_ui(Path.cwd())
